In [ ]:
!pip install pymorphy3

In [ ]:
!pip install ruwordnet

In [ ]:
!ruwordnet download

In [ ]:
import os
import requests
import pandas as pd
import numpy as np

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

In [ ]:
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import pymorphy3
from typing import List, Tuple, Dict, Optional
from torch.cuda.amp import autocast, GradScaler

In [ ]:

def load_russian_lemmas(lemma_file="russian_lemmas.txt"):
    with open(lemma_file, "r", encoding="utf-8") as f:
        lemmas = [line.strip() for line in f if line.strip()]
    return lemmas

RUSSIAN_LEMMAS = load_russian_lemmas()

In [ ]:
morph = pymorphy3.MorphAnalyzer()

def get_word_pos(word: str) -> Optional[str]:
    try:
        parsed = morph.parse(word)[0]
        return parsed.tag.POS
    except:
        return None

In [ ]:
def prepare_data_bert(file_path: str, test_size: float = 0.2, random_state: int = 42):
    df = pd.read_csv(file_path)

    df['input_str'] = df['LFFUNC'] + ' ' + df['LFARG'] + ' [POS_' + df['POS_arg'] + ']'
    df['target_str'] = df['LFVAL']

    # allowed POS per function from value POS
    func_pos_dict = {}
    for func in df['LFFUNC'].unique():
        pos_set = set(df[df['LFFUNC'] == func]['POS_val'].dropna().unique())
        func_pos_dict[func] = pos_set

    train_df, val_df = train_test_split(df, test_size=test_size, random_state=random_state)
    return train_df, val_df, func_pos_dict

In [ ]:
class DualEncoderDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int = 32):
        self.input_texts = df['input_str'].tolist()
        self.target_texts = df['target_str'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.input_texts)

    def __getitem__(self, idx):
        return {
            'input_text': str(self.input_texts[idx]),
            'target_text': str(self.target_texts[idx])
        }

def collate_fn(batch, tokenizer, max_length):
    input_texts = [item['input_text'] for item in batch]
    target_texts = [item['target_text'] for item in batch]

    input_enc = tokenizer(
        input_texts,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )
    target_enc = tokenizer(
        target_texts,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )

    return {
        'input_ids': input_enc['input_ids'],
        'input_attention_mask': input_enc['attention_mask'],
        'target_ids': target_enc['input_ids'],
        'target_attention_mask': target_enc['attention_mask']
    }


In [ ]:
class BertDualEncoder(nn.Module):
    def __init__(self, model_name, projection_dim=None):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.bert.config.hidden_size
        if projection_dim:
            self.projection = nn.Linear(self.hidden_size, projection_dim)
            self.output_dim = projection_dim
        else:
            self.projection = nn.Identity()
            self.output_dim = self.hidden_size

    def encode_input(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_vec = outputs.last_hidden_state[:, 0, :]
        return self.projection(cls_vec)

    def encode_target(self, input_ids, attention_mask):
        return self.encode_input(input_ids, attention_mask)

    def forward(self, input_ids, input_attention_mask, target_ids, target_attention_mask):
        input_vecs = self.encode_input(input_ids, input_attention_mask)
        target_vecs = self.encode_target(target_ids, target_attention_mask)
        return input_vecs, target_vecs

In [ ]:
def contrastive_loss(input_vecs, target_vecs, temperature=0.05):
    input_vecs = F.normalize(input_vecs, p=2, dim=1)
    target_vecs = F.normalize(target_vecs, p=2, dim=1)
    logits = torch.matmul(input_vecs, target_vecs.T) / temperature
    labels = torch.arange(logits.size(0), device=logits.device)
    return F.cross_entropy(logits, labels)


In [ ]:
def train_epoch(model, dataloader, optimizer, device, accumulation_steps=4, temperature=0.05):

    model.train()
    total_loss = 0
    scaler = torch.amp.GradScaler('cuda')
    optimizer.zero_grad()

    progress_bar = tqdm(dataloader, desc='Training', leave=False)
    for i, batch in enumerate(progress_bar):

        input_ids = batch['input_ids'].to(device)
        input_attn = batch['input_attention_mask'].to(device)
        target_ids = batch['target_ids'].to(device)
        target_attn = batch['target_attention_mask'].to(device)

        with torch.amp.autocast('cuda'):
            input_vecs, target_vecs = model(input_ids, input_attn, target_ids, target_attn)
            loss = contrastive_loss(input_vecs, target_vecs, temperature)
            loss = loss / accumulation_steps

        scaler.scale(loss).backward()

        if (i + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * accumulation_steps
        progress_bar.set_postfix({'loss': total_loss / (progress_bar.n + 1)})

    if (i + 1) % accumulation_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    return total_loss / len(dataloader)

In [ ]:
def evaluate(model, dataloader, device, temperature=0.05):

    model.eval()
    total_loss = 0

    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc='Evaluating', leave=False)
        for batch in progress_bar:

            input_ids = batch['input_ids'].to(device)
            input_attn = batch['input_attention_mask'].to(device)
            target_ids = batch['target_ids'].to(device)
            target_attn = batch['target_attention_mask'].to(device)

            input_vecs, target_vecs = model(input_ids, input_attn, target_ids, target_attn)
            loss = contrastive_loss(input_vecs, target_vecs, temperature)

            total_loss += loss.item()
            progress_bar.set_postfix({'loss': total_loss / (progress_bar.n + 1)})

    return total_loss / len(dataloader)

In [ ]:
def train_model(train_loader, val_loader, model, optimizer, num_epochs, device,
                accumulation_steps=4, temperature=0.05):
    history = {'train_loss': [], 'val_loss': []}
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        train_loss = train_epoch(model, train_loader, optimizer, device,
                                  accumulation_steps, temperature)
        val_loss = evaluate(model, val_loader, device, temperature)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    return model, history


In [ ]:
@torch.no_grad()
def encode_candidates(model, candidate_words: List[str], tokenizer, device,
                      max_length=16, batch_size=512):
    """
    Encode a list of candidate words in batches to avoid OOM.
    Returns a tensor of shape [num_candidates, dim] (normalized).
    """
    model.eval()
    all_embeddings = []

    for i in range(0, len(candidate_words), batch_size):
        batch_words = candidate_words[i:i+batch_size]
        encodings = tokenizer(
            batch_words,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt'
        )
        input_ids = encodings['input_ids'].to(device)
        attn_mask = encodings['attention_mask'].to(device)

        vecs = model.encode_input(input_ids, attn_mask)
        vecs = F.normalize(vecs, p=2, dim=1)
        all_embeddings.append(vecs.cpu())

    return torch.cat(all_embeddings, dim=0)

def prepare_candidates(model, tokenizer, device, max_length=16, batch_size=512):
    """Precompute embeddings for all Russian words (batched) and store globally."""
    global candidate_embeddings, candidate_words_global
    candidate_words_global = RUSSIAN_LEMMAS
    candidate_embeddings = encode_candidates(
        model, candidate_words_global, tokenizer, device,
        max_length=max_length, batch_size=batch_size
    )

In [ ]:
def lemmatize_word(word: str) -> str:
    try:
        return morph.parse(word)[0].normal_form
    except:
        return word

def predict_lf_bert(model, tokenizer, function: str, word: str,
                    func_pos_dict: Dict, device, top_k: int = 5,
                    max_length: int = 32) -> List[Tuple[str, float]]:

    global candidate_embeddings, candidate_words_global
    if candidate_embeddings is None:
        raise RuntimeError("error - prepare_candidates() not called")

    model.eval()

    lemma_word = lemmatize_word(word)
    pos_arg = get_word_pos(word) or "UNKNOWN"
    input_text = f"{function} {lemma_word} [POS_{pos_arg}]"

    enc = tokenizer(
        input_text,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )
    input_ids = enc['input_ids'].to(device)
    attn_mask = enc['attention_mask'].to(device)

    with torch.no_grad():
        query_vec = model.encode_input(input_ids, attn_mask)
        query_vec = F.normalize(query_vec, p=2, dim=1)

    query_vec_cpu = query_vec.detach().cpu()

    sims = torch.matmul(query_vec_cpu, candidate_embeddings.T).squeeze(0).numpy()

    allowed_pos = func_pos_dict.get(function, set())
    candidates_with_pos = []
    for word_cand, score in zip(candidate_words_global, sims):
        first_word = word_cand.split()[0] if word_cand else ''
        pos_val = get_word_pos(first_word)
        if allowed_pos and pos_val not in allowed_pos:
            continue
        candidates_with_pos.append((word_cand, score))

    candidates_with_pos.sort(key=lambda x: -x[1])
    return candidates_with_pos[:top_k]

In [ ]:

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'ai-forever/ruBert-large'
BATCH_SIZE = 8
MAX_LENGTH = 32
EPOCHS = 6
LR = 2e-5
ACCUMULATION_STEPS = 4
TEMPERATURE = 0.05
PROJECTION_DIM = 256

file_path = "lf_data_words_with_pos.csv"

train_df, val_df, func_pos_dict = prepare_data_bert(file_path)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = DualEncoderDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = DualEncoderDataset(val_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, tokenizer, MAX_LENGTH),
    num_workers=2,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda b: collate_fn(b, tokenizer, MAX_LENGTH),
    num_workers=2,
    pin_memory=True
)

model = BertDualEncoder(MODEL_NAME, projection_dim=PROJECTION_DIM).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR)

model, history = train_model(
    train_loader,
    val_loader,
    model,
    optimizer,
    num_epochs=EPOCHS,
    device=DEVICE,
    accumulation_steps=ACCUMULATION_STEPS,
    temperature=TEMPERATURE
)

torch.save(model.state_dict(), './dual_encoder_rubert.pt')
tokenizer.save_pretrained('./dual_encoder_tokenizer')

prepare_candidates(model, tokenizer, DEVICE, max_length=16)

test_cases_old = [
    ('MAGN', 'довод'),
    ('OPER1', 'домино'),
    ('OPER2', 'арест'),
    ('INCEPOPER1', 'азарт'),
    ('FUNC0', 'дорога'),
    ('INCEPFUNC0', 'день'),
    ('CAUSFUNC0', 'встреча'),
    ('REAL1', 'газета'),
    ('REAL1-M', 'долг')
]
test_cases_new = [
    ('LOC', 'дом'),
    ('OPER1', 'оценка'),
    ('MAGN', 'друг'),
    ('ADV2-UN', 'причина'),
    ('ADV1-UN', 'надежда'),
    ('CAUSFUNC0', 'заседание'),
    ('FUNC0', 'дело'),
    ('INCEPOPER1', 'право'),
    ('OPER2', 'внимание')
]
all_test_cases = test_cases_old + test_cases_new

for test_func, test_word in all_test_cases:
    predictions = predict_lf_bert(
        model, tokenizer, test_func, test_word,
        func_pos_dict=func_pos_dict,
        device=DEVICE,
        top_k=5
    )
    print(f"\n{test_func}({test_word}):")
    if not predictions:
        print("  No predictions after POS filtering.")
    else:
        for i, (val, score) in enumerate(predictions, 1):
            pos_val = get_word_pos(val.split()[0]) if val else ''
            print(f"  {i}. {val} (score: {score:.4f}, POS: {pos_val})")